In [ ]:
# Install required libraries
!pip install torch rdkit pandas scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 16.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
##Code generated by AI

import pandas as pd

# Path to your dataset
dataset_path = "/content/drive/MyDrive/Projects/NeurIPS Polymer Prediction 2025/data/train.csv"

# Load the data
df = pd.read_csv(dataset_path)
df = df[df['SMILES'].notna() & df['FFV'].notna()] # Remove rows with missing labels for simplicity
df.head()


,id,SMILES,Tg,FFV,Tc,Density,Rg
0,87817,*CC(*)c1ccccc1C(=O)OCCCCCC,NaN,0.374645,0.205667,NaN,NaN
1,106919,*Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5...,NaN,0.370410,NaN,NaN,NaN
2,388772,*Oc1ccc(S(=O)(=O)c2ccc(Oc3ccc(C4(c5ccc(Oc6ccc(...,NaN,0.378860,NaN,NaN,NaN
3,519416,*Nc1ccc(-c2c(-c3ccc(C)cc3)c(-c3ccc(C)cc3)c(N*)...,NaN,0.387324,NaN,NaN,NaN
4,539187,*Oc1ccc(OC(=O)c2cc(OCCCCCCCCCOCC3CCCN3c3ccc([N...,NaN,0.355470,NaN,NaN,NaN


In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.9 MB/s eta 0:00:00


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import torch
from torch_geometric.data import Data

def atom_features(atom):
    return torch.tensor([
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetHybridization().real,
        atom.GetIsAromatic()
    ], dtype=torch.float)

def bond_features(bond):
    return torch.tensor([
        bond.GetBondTypeAsDouble(),
        bond.IsInRing()
    ], dtype=torch.float)

def smiles_to_graph(smiles, y):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    x = torch.stack([atom_features(atom) for atom in mol.GetAtoms()])
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index += [[i, j], [j, i]]
        edge_attr += [bond_features(bond), bond_features(bond)]

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.stack(edge_attr)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=torch.tensor([y], dtype=torch.float))


In [ ]:
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

graphs = []
for _, row in df.iterrows():
    data = smiles_to_graph(row['SMILES'], row['FFV'])
    if data is not None:
        graphs.append(data)

train_graphs, test_graphs = train_test_split(graphs, test_size=0.1, random_state=42)
train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=32)


In [ ]:
import torch.nn as nn
from torch_geometric.nn import GCNConv, global_mean_pool

class GNNRegressor(nn.Module):
    def __init__(self, node_feat_dim, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(node_feat_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin1 = nn.Linear(hidden_dim, 64)
        self.lin2 = nn.Linear(64, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = torch.relu(self.conv1(x, edge_index))
        x = torch.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        x = torch.relu(self.lin1(x))
        return self.lin2(x)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
model = GNNRegressor(node_feat_dim=train_graphs[0].x.size(1)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def test():
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            pred = model(batch)
            preds.append(pred.cpu())
            truths.append(batch.y.cpu())
    return torch.cat(preds), torch.cat(truths)


In [ ]:
best_mse = float('inf')
best_model_path = "/content/drive/MyDrive/Projects/NeurIPS Polymer Prediction 2025/models/FVV_best_density_model.pth"

for epoch in range(1000):
    loss = train()
    preds, truths = test()
    mse = torch.mean(torch.abs(preds - truths)**2).item()
    print(f"Epoch {epoch:02d} | Train Loss: {loss:.4f} | Test MSE: {mse:.4f}")
    if mse < best_mse:
        best_mse = mse
        torch.save(model.state_dict(), best_model_path)
        print(f"New best model saved (MSE: {mse:.4f})")


Epoch 00 | Train Loss: 0.0010 | Test MSE: 0.0013
New best model saved (MAE: 0.0013)
Epoch 01 | Train Loss: 0.0010 | Test MSE: 0.0011
New best model saved (MAE: 0.0011)
Epoch 02 | Train Loss: 0.0010 | Test MSE: 0.0011
New best model saved (MAE: 0.0011)
Epoch 03 | Train Loss: 0.0010 | Test MSE: 0.0011
New best model saved (MAE: 0.0011)
Epoch 04 | Train Loss: 0.0010 | Test MSE: 0.0011
Epoch 05 | Train Loss: 0.0009 | Test MSE: 0.0011
New best model saved (MAE: 0.0011)
Epoch 06 | Train Loss: 0.0009 | Test MSE: 0.0011
Epoch 07 | Train Loss: 0.0009 | Test MSE: 0.0011
New best model saved (MAE: 0.0011)
Epoch 08 | Train Loss: 0.0009 | Test MSE: 0.0011
New best model saved (MAE: 0.0011)
Epoch 09 | Train Loss: 0.0009 | Test MSE: 0.0011
Epoch 10 | Train Loss: 0.0009 | Test MSE: 0.0011
New best model saved (MAE: 0.0011)
Epoch 11 | Train Loss: 0.0009 | Test MSE: 0.0011
Epoch 12 | Train Loss: 0.0009 | Test MSE: 0.0011
Epoch 13 | Train Loss: 0.0009 | Test MSE: 0.0011
New best model saved (MAE: 0.0011)